# Final evaluation и inference

Финальная V2 используется для демонстрации предсказаний на новых названиях товаров

Веса не входят в репозиторий: ожидается `models/v2_weighted`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from collections import Counter
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from src.project_utils import (
    SEED, MODEL_NAME, DATASET_NAME, NUM_LABELS, MAX_LENGTH,
    set_seed, clean_html, build_text, build_text_with_attributes,
    compute_metrics, make_model, WeightedTrainer, predict_category,
)

set_seed(SEED)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.14.0+cu126
CUDA: True


In [2]:
dataset = load_dataset(DATASET_NAME)
target_column = "greatgrandparent_category"
dataset = dataset.filter(lambda example: example[target_column] is not None)
label_names = sorted(set(dataset["train"][target_column]))
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
V2_DIR = Path("../models/v2_weighted")
final_model = AutoModelForSequenceClassification.from_pretrained(V2_DIR)
print("Loaded:", V2_DIR)

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 10833.08it/s]

Loaded: ..\models\v2_weighted


## Inference function

In [3]:
examples = [
    "Смартфон Samsung Galaxy A55 256 ГБ, 8 ГБ RAM",
    "Набор керамических тарелок 6 штук",
    "Женские зимние ботинки натуральная кожа",
    "Корм для кошек сухой, 10 кг",
]

for text in examples:
    print(f"{text} -> {predict_category(text, final_model, tokenizer, label_names)}")

Смартфон Samsung Galaxy A55 256 ГБ, 8 ГБ RAM -> Электроника
Набор керамических тарелок 6 штук -> Товары для кухни
Женские зимние ботинки натуральная кожа -> Обувь
Корм для кошек сухой, 10 кг -> Зоотовары


## Итог проекта

Финальная модель — `rubert-tiny2` с weighted Cross-Entropy. На test получены Accuracy **0.9734**, Macro F1 **0.9493**, Weighted F1 **0.9734**. Все исходные эксперименты и результаты зафиксированы в ноутбуках проекта